# What each allowance costs us

The plan table as a subscriber reads it at [`/app/plans`](../client/src/pages/PlanPricingPage.tsx)
— every metered service as a row, every tier as a column — with **what we pay**
added to each cell beside the allowance it buys.

Every cell answers one question: *if this subscriber spent this allowance down to
zero, what do we owe the vendor for it?* That is the worst case, not the expected
one; [`billing-cost-model.ipynb`](billing-cost-model.ipynb) is where the same caps
get modelled at realistic utilisation, priced against fixed costs, and turned into
break-even subscriber counts.

**Inputs**, all read rather than copied, so this describes what actually ships:

| Source | What comes from it |
| --- | --- |
| [`pricing.json`](pricing.json) | vendor unit prices (a working copy of `config/service-prices.json` — drift-checked below) |
| [`assumptions.json`](assumptions.json) | the usage assumptions behind the derived rates (deck size, export size) |
| [`../config/plans.json`](../config/plans.json) | the shipped caps, tier by tier |
| [`../client/src/i18n/locales/en.json`](../client/src/i18n/locales/en.json) | the row labels and hints, so this table says what the page says |
| `../server/src/billing/usage-view.ts`, `../shared/src/dto/usage.ts` | the row order and the instructor/audience split |
| [`../scripts/pricing/derive-caps.mjs`](../scripts/pricing/derive-caps.mjs) | §6 calls it to show where eight of the caps come from |

## 1. Load the inputs

In [1]:
# Nothing is computed here — this only reads the shipped configuration, the
# translated labels the pricing page renders, and the modelling inputs.
import json, os, re
from IPython.display import Markdown, display

ROOT = os.path.abspath('..')             # repo root, one level up from cost-model/
P = json.load(open('pricing.json'))      # vendor unit prices (modelling copy)
A = json.load(open('assumptions.json'))  # usage assumptions
PLANS = json.load(open(os.path.join(ROOT, 'config', 'plans.json')))
STRINGS = json.load(open(os.path.join(ROOT, 'client', 'src', 'i18n', 'locales', 'en.json')))

TIERS = list(PLANS)                      # 'free', 'fresh', 'pro', 'max' — cheapest first


def s(path, **vars):
    """Looks up a dotted i18n key in the English bundle, e.g. 'usage.metric.aiTokens'.

    Reading the shipped bundle rather than retyping the labels means a row here
    can never end up named something the pricing page does not call it. Missing
    keys return the key itself, which is visible enough to notice.
    """
    node = STRINGS
    for part in path.split('.'):
        if not isinstance(node, dict) or part not in node:
            return path
        node = node[part]
    if not isinstance(node, str):
        return path
    for k, v in vars.items():
        node = node.replace('{' + k + '}', str(v))
    return node


def plural(path, count):
    """Resolves one of the bundle's ICU plural strings — '{count, plural, one
    {# day} other {# days}}' — for a count, substituting '#'. English only,
    which is all this notebook renders."""
    message = s(path)
    form = re.search(r'\b(one)\s*\{([^{}]*)\}', message) if count == 1 else None
    form = form or re.search(r'\bother\s*\{([^{}]*)\}', message)
    if not form:
        return message
    return form.groups()[-1].replace('#', f'{count:,.0f}')


def table(headers, rows, title=None):
    """Renders rows as a markdown table. Jupyter displays markdown output, so
    this gives a readable table without pandas — any Python 3 kernel will do."""
    out = (f'**{title}**\n\n' if title else '')
    out += '| ' + ' | '.join(headers) + ' |\n'
    out += '|' + '|'.join(['---'] * len(headers)) + '|\n'
    for r in rows:
        out += '| ' + ' | '.join(str(c) for c in r) + ' |\n'
    display(Markdown(out))


def usd(x, dp=2):
    """Formats dollars, sign outside the symbol, and never rounds a real cost
    down to a bare '$0.00' — several of these rates are fractions of a cent."""
    sign = '-' if x < 0 else ''
    x = abs(x)
    if x and x < 10 ** -dp / 2:
        return f'{sign}<${10 ** -dp:,.{dp}f}'
    return f'{sign}${x:,.{dp}f}'


print(f"pricing asOf {P['asOf']} | tiers {TIERS} | default model {P['ai']['defaultModel']}")

pricing asOf 2026-07-31 | tiers ['free', 'fresh', 'pro', 'max'] | default model gemini-3.1-flash-lite-preview


### Drift check

`pricing.json` is the modelling copy, so it can be experimented with freely. The
server bills against [`../config/service-prices.json`](../config/service-prices.json).
If the two have diverged, every dollar below describes a product we do not sell.

In [2]:
# Walk both price files in parallel and report any leaf that differs. Loud on
# purpose: a silent drift here turns this notebook into fiction.
live = json.load(open(os.path.join(ROOT, 'config', 'service-prices.json')))


def diffs(a, b, path=''):
    """Yields (path, modelling value, shipped value) for every differing leaf."""
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b)):
            if k.startswith('_'):
                continue          # _sources and _note are commentary, not prices
            yield from diffs(a.get(k), b.get(k), f'{path}.{k}' if path else k)
    elif a != b:
        yield path, a, b


drift = list(diffs(P, live))
if drift:
    table(['Key', 'pricing.json', 'config/service-prices.json'], drift,
          '⚠️ Prices differ — reconcile before trusting anything below')
else:
    print('✅ pricing.json matches config/service-prices.json')

✅ pricing.json matches config/service-prices.json


## 2. Rows, in the order the page shows them

The pricing page does not choose its own row order: it sorts by
`metricSortKey` — instructor allowances first, then the audience pool, and
within each the fixed `ORDER` list. Both are parsed out of the shipped
TypeScript here rather than retyped, so a metric added or reordered there
moves here too.

In [3]:
def ts_string_array(rel_path, declaration):
    """Pulls a `const NAME ... = ['a', 'b']` array of string literals out of a
    TypeScript file. Cheap by design — these are two hand-maintained lists of
    plain identifiers, not something worth a parser."""
    src = open(os.path.join(ROOT, *rel_path.split('/'))).read()
    body = re.search(re.escape(declaration) + r'[^=]*=\s*\[(.*?)\]', src, re.S)
    if not body:
        raise SystemExit(f'could not find {declaration} in {rel_path} — has it been renamed?')
    return re.findall(r"'([A-Za-z]+)'", body.group(1))


# Display order, and which metrics belong to the audience's separate pool.
ORDER = ts_string_array('server/src/billing/usage-view.ts', 'const ORDER')
AUDIENCE_METRICS = ts_string_array('shared/src/dto/usage.ts', 'const AUDIENCE_METRICS')

# Units, so a cap can be labelled the way the account's own meters label it.
UNITS = {
    'aiTokens': 'tokens', 'sttMinutes': 'minutes', 'diarizationMinutes': 'minutes',
    'ttsCharacters': 'characters', 'ttsPremiumCharacters': 'characters',
    'translationCharacters': 'characters', 'audienceTtsCharacters': 'characters',
    'importMb': 'mb', 'audioStorageMb': 'mb',
}

allowance_of = lambda m: 'audience' if m in AUDIENCE_METRICS else 'instructor'
# Mirrors metricSortKey: group first, then position in ORDER, unknowns last.
sort_key = lambda m: (allowance_of(m) == 'audience',
                      ORDER.index(m) if m in ORDER else len(ORDER))

# Every metric any tier caps, which is every metric in PlanCaps.
METRICS = sorted({m for p in PLANS.values() for m in p['caps']}, key=sort_key)

print(f'{len(METRICS)} metered rows: '
      f"{sum(1 for m in METRICS if allowance_of(m) == 'instructor')} instructor, "
      f"{sum(1 for m in METRICS if allowance_of(m) == 'audience')} audience")

13 metered rows: 11 instructor, 2 audience


## 3. What one unit of each allowance costs

One rate per row: what we pay the vendor for a single token, minute, character,
image, megabyte or export. Every cost in the final table is this rate times the
cap — the caps are all linear in what they buy, so nothing else is needed.

Three rows are **derived** rather than quoted, and are only as good as the
assumption under them:

- **Translations for viewers** is metered in whole decks, so it is priced at a
  deck's worth of slide text — `slidesPerLecture × slideTextChars`.
- **Exports** is metered in operations; what an export costs is the egress of
  the deck's images and audio leaving object storage.
- **AI generation** blends the input and output token rates, since one cap
  covers both. Input dominates at roughly 93% of tokens spent.

And two rows genuinely cost us **nothing**, which is worth stating rather than
leaving as a suspicious zero:

- **Image searches** run against Wikimedia, Openverse and Flickr — keyless,
  free APIs (`server/src/enrichment/gather.ts`). The cap exists to bound our
  fan-out and stay a good citizen of somebody else's rate limit, not to recover
  a cost.
- **Original audio retention** has no rate of its own; the audio it keeps is
  already priced by the *Stored audio* row.

In [4]:
# The configured model and voice families, not today's names hard-coded: change
# defaultModel or defaultPremiumFamily in pricing.json and this re-prices.
model = P['ai']['models'][P['ai']['defaultModel']]

# Vendors quote per million; caps count single units. Divide once here so every
# rate below is simply 'price per one of whatever the cap counts'.
BLEND_INPUT = 0.93          # share of tokens that are prompt rather than completion
AI_BLENDED = (BLEND_INPUT * model['inputPerMillionTokens']
              + (1 - BLEND_INPUT) * model['outputPerMillionTokens']) / 1e6

# Speech-to-Text V2 bills streaming and standard batch at one rate, so
# diarization — a batch job — costs the same per minute as live capture.
STT = P['stt']['recognitionPerMinute']
TTS_STD = P['tts']['voiceFamilies'][P['tts']['defaultStandardFamily']]['perMillionChars'] / 1e6
TTS_PREM = P['tts']['voiceFamilies'][P['tts']['defaultPremiumFamily']]['perMillionChars'] / 1e6
TRANSLATE = P['translation']['perMillionChars'] / 1e6
IMAGE = list(P['ai']['imageModels'].values())[0]['perImage']
GIB_MONTH, EGRESS = P['storage']['perGibMonth'], P['storage']['egressPerGib']
PAY_RATE = P['payments']['rate'] + P['payments']['billingRate']
PAY_FIXED = P['payments']['perTransaction']

# --- the three derived rates -------------------------------------------------
# One viewer-requested locale translates one deck's worth of slide text.
DECK_TEXT_CHARS = A['lecture']['slidesPerLecture'] * A['lecture']['slideTextChars']
# One export ships the deck's images and narration audio out of storage.
DECK_MB = A['audience']['imageMbPerPlayback'] + A['audience']['narrationAudioMbPerPlayback']
MB_MONTH = GIB_MONTH / 1024   # storage is quoted per GiB; two caps are in MB

# Rate per unit of the cap, and how to say it out loud. The label is what the
# row header shows, so the multiplication in each cell can be checked by eye.
RATES = {
    'sttMinutes':            (STT,                        f'{usd(STT, 3)} / min'),
    'aiTokens':              (AI_BLENDED,                  f'{usd(AI_BLENDED * 1e6)} / 1M tokens, blended {BLEND_INPUT:.0%} input'),
    'ttsCharacters':         (TTS_STD,                     f"{usd(TTS_STD * 1e6)} / 1M chars ({P['tts']['defaultStandardFamily']})"),
    'ttsPremiumCharacters':  (TTS_PREM,                    f"{usd(TTS_PREM * 1e6)} / 1M chars ({P['tts']['defaultPremiumFamily']})"),
    'diarizationMinutes':    (STT,                         f'{usd(STT, 3)} / min'),
    'translationCharacters': (TRANSLATE,                   f'{usd(TRANSLATE * 1e6)} / 1M chars'),
    'aiImages':              (IMAGE,                       f'{usd(IMAGE, 4)} / image'),
    'imageLookups':          (0.0,                         'free APIs — no vendor charge'),
    'audioStorageMb':        (MB_MONTH,                    f'{usd(GIB_MONTH, 3)} / GiB-month held'),
    'importMb':              (MB_MONTH,                    f'{usd(GIB_MONTH, 3)} / GiB-month held'),
    'exports':               (DECK_MB / 1024 * EGRESS,     f'{usd(EGRESS, 3)} / GiB egress × {DECK_MB:.1f} MB per deck'),
    'audienceTtsCharacters': (TTS_STD,                     f"{usd(TTS_STD * 1e6)} / 1M chars ({P['tts']['defaultStandardFamily']})"),
    'audienceLocales':       (DECK_TEXT_CHARS * TRANSLATE, f'{usd(TRANSLATE * 1e6)} / 1M chars × {DECK_TEXT_CHARS:,.0f} chars per deck'),
}

missing = [m for m in METRICS if m not in RATES]
if missing:
    print(f'⚠️ no rate for {missing} — these rows will price at zero')

table(['Row', 'We pay', 'Per unit of'],
      [[s(f'usage.metric.{m}'), RATES[m][1], UNITS.get(m, 'count')] for m in METRICS],
      'Unit cost behind every cell below')

**Unit cost behind every cell below**

| Row | We pay | Per unit of |
|---|---|---|
| Audio recording time | $0.016 / min | minutes |
| AI generation | $0.34 / 1M tokens, blended 93% input | tokens |
| Narration | $16.00 / 1M chars (neural2) | characters |
| Premium narration | $30.00 / 1M chars (chirp3-hd) | characters |
| Speaker identification | $0.016 / min | minutes |
| Translation | $20.00 / 1M chars | characters |
| AI images | $0.0336 / image | count |
| Image searches | free APIs — no vendor charge | count |
| Stored audio | $0.020 / GiB-month held | mb |
| Imports | $0.020 / GiB-month held | mb |
| Exports | $0.010 / GiB egress × 14.4 MB per deck | count |
| Narration for viewers | $16.00 / 1M chars (neural2) | characters |
| Translations for viewers | $20.00 / 1M chars × 18,000 chars per deck | count |


## 4. Saying a cap the way the page says it

A cap is stored in the unit we bill in and shown in the unit a reader thinks in:
narration allowances become minutes of speech, translation becomes words, token
allowances become millions. This is a port of `friendlyCap` and `formatAmount`
from [`../client/src/lib/usage.ts`](../client/src/lib/usage.ts), using the same
constants and the same English strings, so the left half of every cell below
matches the page character for character.

In [5]:
CHARACTERS_PER_SPOKEN_MINUTE = 900   # ~150 words/min at ~6 chars a word
CHARACTERS_PER_WORD = 6
MINUTES_AS_HOURS = 120               # past this, an allowance reads better in hours

# Which metrics are re-expressed, and how. The rest are already in units a
# reader recognises — minutes recorded, images, exports, megabytes.
FRIENDLY = {
    'ttsCharacters': 'spoken', 'ttsPremiumCharacters': 'spoken',
    'audienceTtsCharacters': 'spoken', 'translationCharacters': 'words',
    'audienceLocales': 'languages', 'aiTokens': 'tokens',
}
num = lambda x: f'{x:,.0f}'


def cap_label(metric, cap):
    """A cap as the pricing page prints it — 'about 67 min of narration',
    'Unlimited', 'Not included' for the 0 sentinel no shipped tier uses."""
    if cap is None:
        return s('plan.pricing.unlimited')
    if cap == 0:
        return s('plan.pricing.notIncluded')
    kind = FRIENDLY.get(metric)
    if kind == 'languages':
        return plural('plan.pricing.approx.languages', cap)
    if kind == 'tokens':
        millions = cap / 1e6
        shown = f'{millions:,.0f}' if float(millions).is_integer() else f'{round(millions * 10) / 10:,.1f}'
        return s('plan.pricing.approx.tokens', value=shown)
    if kind == 'words':
        return s('plan.pricing.approx.words', value=num(round(cap / CHARACTERS_PER_WORD)))
    if kind == 'spoken':
        minutes = cap / CHARACTERS_PER_SPOKEN_MINUTE
        return (s('plan.pricing.approx.spokenHours', value=num(round(minutes / 60)))
                if minutes >= MINUTES_AS_HOURS
                else s('plan.pricing.approx.spokenMinutes', value=num(round(minutes))))
    unit = UNITS.get(metric, 'count')
    if unit == 'minutes':
        return s('usage.unit.minutes', value=num(cap))
    if unit == 'mb':
        return s('usage.unit.mb', value=num(cap))
    return num(cap)


def raw_label(metric, cap):
    """The metered quantity behind a friendly label — '60,000 chars'. The page
    has no reason to show this; a cost model does, because it is the number the
    rate is multiplied by."""
    unit = UNITS.get(metric, 'count')
    if cap is None or cap == 0 or unit in ('minutes', 'mb', 'count', 'tokens'):
        return None      # already shown as-is, or already legible ('5M tokens')
    return f'{num(cap)} chars'


# Spot-check against the values the page renders for the Free tier.
free = PLANS['free']['caps']
print(' | '.join(f"{s(f'usage.metric.{m}')}: {cap_label(m, free[m])}"
                 for m in ('aiTokens', 'ttsCharacters', 'translationCharacters', 'audienceLocales')))

AI generation: 5M tokens | Narration: about 67 min of narration | Translation: about 1,500 words | Translations for viewers: 1 language


## 5. The table

The pricing page's two metered bands — **Your allowances** and **Your
audience** — with the same rows in the same order, the same labels and hints,
retention still sitting under the recording allowance it qualifies. Each cell
carries the allowance the page shows, the metered quantity behind it where that
differs, and **what it costs us if the subscriber spends all of it**.

Read a column downwards for what one maxed-out subscriber on that tier can cost;
read a row across for which tier makes a service expensive.

In [6]:
SHOW_RAW = True     # set False for cells that read exactly like the page

cost_of = lambda metric, cap: (cap or 0) * RATES.get(metric, (0.0, ''))[0]


def cell(metric, cap):
    """One data cell: the allowance, the raw quantity behind it, the cost."""
    parts = [cap_label(metric, cap)]
    raw = raw_label(metric, cap) if SHOW_RAW else None
    if raw:
        parts.append(f'<sub>{raw}</sub>')
    parts.append(f'**{usd(cost_of(metric, cap))}**')
    return '<br>'.join(parts)


def row_header(label, hint, rate=None):
    """A row's name, what the allowance covers, and what we pay per unit — the
    page's name-and-hint header with the rate appended."""
    out = f'**{label}**'
    if hint:
        out += f'<br><sub>{hint}</sub>'
    if rate:
        out += f'<br><sub>💸 {rate}</sub>'
    return out


def retention_row():
    """How long recordings are kept. A policy rather than a meter, and priced by
    the Stored audio row above it — but it is what makes that row's cost recur,
    so it is shown where the page shows it."""
    cells = []
    for tier in TIERS:
        days = PLANS[tier]['audioRetentionDays']
        cells.append(s('plan.pricing.retentionUnlimited') if days is None
                     else plural('plan.pricing.retentionDays', days))
    return [row_header(s('plan.pricing.audioRetention'),
                       s('plan.metricHint.audioRetention'),
                       'no charge of its own — priced by Stored audio')] + cells


def band(allowance, title, hint=None):
    """One titled band of rows, plus its subtotal. Returns (rows, totals)."""
    metrics = [m for m in METRICS if allowance_of(m) == allowance]
    rows = [[f'**{title.upper()}**'] + [''] * len(TIERS)]
    if hint:
        rows.append([f'<sub>{hint}</sub>'] + [''] * len(TIERS))
    for metric in metrics:
        rows.append([row_header(s(f'usage.metric.{metric}'),
                                s(f'plan.metricHint.{metric}'),
                                RATES.get(metric, (0, None))[1])]
                    + [cell(metric, PLANS[t]['caps'].get(metric)) for t in TIERS])
        # Retention qualifies the recording allowance, so it follows it — as on
        # the page, which puts it directly under sttMinutes.
        if metric == 'sttMinutes':
            rows.append(retention_row())
    totals = {t: sum(cost_of(m, PLANS[t]['caps'].get(m)) for m in metrics) for t in TIERS}
    rows.append([f'*{title} subtotal*'] + [f'**{usd(totals[t])}**' for t in TIERS])
    return rows, totals


def price_of(tier):
    """The tier's monthly price. plans.json holds only the provider's price id,
    so the amount comes from the modelling assumptions."""
    return A['tiers'][tier]['priceUsd']


header = ['&nbsp;'] + [f"**{s('plan.tier.' + t)}**<br><sub>"
                       + (s('plan.pricing.free') if not price_of(t)
                          else f'${price_of(t):,.0f} per month') + '</sub>'
                       for t in TIERS]

instructor_rows, instructor_total = band('instructor', s('usage.instructor'))
audience_rows, audience_total = band('audience', s('usage.audience'), s('usage.audienceHint'))
maxed = {t: instructor_total[t] + audience_total[t] for t in TIERS}

summary = [
    ['**TOTAL — every allowance spent**'] + [f'**{usd(maxed[t])}**' for t in TIERS],
    ['<sub>as a share of the price</sub>']
    + [f"<sub>{maxed[t] / price_of(t) * 100:.0f}% of ${price_of(t):,.0f}</sub>"
       if price_of(t) else '<sub>—</sub>' for t in TIERS],
    ['<sub>left after payment fees</sub>']
    + [f'<sub>{usd(price_of(t) * (1 - PAY_RATE) - PAY_FIXED - maxed[t])}</sub>'
       if price_of(t) else f'<sub>{usd(-maxed[t])}</sub>' for t in TIERS],
]

table(header, instructor_rows + audience_rows + summary,
      'Maximum vendor cost per subscriber per period, at the shipped caps')

**Maximum vendor cost per subscriber per period, at the shipped caps**

| &nbsp; | **Free**<br><sub>No charge</sub> | **Fresh**<br><sub>$19 per month</sub> | **Pro**<br><sub>$99 per month</sub> | **Max**<br><sub>$299 per month</sub> |
|---|---|---|---|---|
| **YOUR ALLOWANCES** |  |  |  |  |
| **Audio recording time**<br><sub>Live speech transcribed while you lecture.</sub><br><sub>💸 $0.016 / min</sub> | 75 min<br>**$1.20** | 150 min<br>**$2.40** | 600 min<br>**$9.60** | 3,300 min<br>**$52.80** |
| **Original audio retention**<br><sub>How long the original recordings are kept before they are deleted.</sub><br><sub>💸 no charge of its own — priced by Stored audio</sub> | 7 days | 14 days | 21 days | Kept indefinitely |
| **AI generation**<br><sub>AI work across slide generation, refining, narration and quizzes. Metered in tokens: one slide-generation request runs about 3,500, and a 75-minute lecture around 2M.</sub><br><sub>💸 $0.34 / 1M tokens, blended 93% input</sub> | 5M tokens<br>**$1.69** | 7.5M tokens<br>**$2.53** | 65M tokens<br>**$21.94** | 110M tokens<br>**$37.12** |
| **Narration**<br><sub>Narration you generate with a standard voice. Counted once, when a slide is first narrated — replaying it costs nothing.</sub><br><sub>💸 $16.00 / 1M chars (neural2)</sub> | about 67 min of narration<br><sub>60,000 chars</sub><br>**$0.96** | about 2 h of narration<br><sub>120,000 chars</sub><br>**$1.92** | about 15 h of narration<br><sub>800,000 chars</sub><br>**$12.80** | about 26 h of narration<br><sub>1,400,000 chars</sub><br>**$22.40** |
| **Premium narration**<br><sub>Narration generated with the more natural premium voices. Counted once, when a slide is first narrated — replaying it costs nothing.</sub><br><sub>💸 $30.00 / 1M chars (chirp3-hd)</sub> | about 22 min of narration<br><sub>20,000 chars</sub><br>**$0.60** | about 44 min of narration<br><sub>40,000 chars</sub><br>**$1.20** | about 111 min of narration<br><sub>100,000 chars</sub><br>**$3.00** | about 8 h of narration<br><sub>440,000 chars</sub><br>**$13.20** |
| **Speaker identification**<br><sub>Recorded audio you may label by speaker. Matches your recording allowance, so anything you record can also be labelled — it is a separate pass, run on the lectures that need it.</sub><br><sub>💸 $0.016 / min</sub> | 75 min<br>**$1.20** | 150 min<br>**$2.40** | 600 min<br>**$9.60** | 3,300 min<br>**$52.80** |
| **Translation**<br><sub>Lecture text you translate yourself. Counted once per lecture and language — the translation is stored and reused.</sub><br><sub>💸 $20.00 / 1M chars</sub> | about 1,500 words<br><sub>9,000 chars</sub><br>**$0.18** | about 3,000 words<br><sub>18,000 chars</sub><br>**$0.36** | about 16,667 words<br><sub>100,000 chars</sub><br>**$2.00** | about 66,667 words<br><sub>400,000 chars</sub><br>**$8.00** |
| **AI images**<br><sub>Images generated by AI for your slides.</sub><br><sub>💸 $0.0336 / image</sub> | 5<br>**$0.17** | 10<br>**$0.34** | 100<br>**$3.36** | 300<br>**$10.08** |
| **Image searches**<br><sub>Slide images sourced from image search.</sub><br><sub>💸 free APIs — no vendor charge</sub> | 250<br>**$0.00** | 350<br>**$0.00** | 1,500<br>**$0.00** | 2,200<br>**$0.00** |
| **Stored audio**<br><sub>Retained lecture audio held at any one time.</sub><br><sub>💸 $0.020 / GiB-month held</sub> | 500 MB<br>**$0.01** | 1,000 MB<br>**$0.02** | 8,000 MB<br>**$0.16** | 58,000 MB<br>**$1.13** |
| **Imports**<br><sub>Documents uploaded as source material.</sub><br><sub>💸 $0.020 / GiB-month held</sub> | 100 MB<br>**<$0.01** | 300 MB<br>**$0.01** | 2,000 MB<br>**$0.04** | 10,000 MB<br>**$0.20** |
| **Exports**<br><sub>Downloads, exports to Drive, and quiz publishing.</sub><br><sub>💸 $0.010 / GiB egress × 14.4 MB per deck</sub> | 10<br>**<$0.01** | 30<br>**<$0.01** | 300<br>**$0.04** | 1,000<br>**$0.14** |
| *Your allowances subtotal* | **$6.01** | **$11.18** | **$62.53** | **$197.87** |
| **YOUR AUDIENCE** |  |  |  |  |
| <sub>Spent by people viewing your lectures, from a separate pool so a popular lecture never uses up your own.</sub> |  |  |  |  |
| **Narration for viewers**<br><sub>Narration your viewers trigger, from a pool of its own so a popular lecture never spends yours. Counted once per slide, however many people listen.</sub><br><sub>💸 $16.00 / 1M chars (neural2)</sub> | about 28 min of narration<br><sub>25,000 chars</sub><br>**$0.40** | about 50 min of narration<br><sub>45,000 chars</sub><br>**$0.72** | about 8 h of narration<br><sub>450,000 chars</sub><br>**$7.20** | about 13 h of narration<br><sub>700,000 chars</sub><br>**$11.20** |
| **Translations for viewers**<br><sub>Languages your viewers may have a lecture translated into. Counted once per language; everyone after the first viewer reads the stored translation.</sub><br><sub>💸 $20.00 / 1M chars × 18,000 chars per deck</sub> | 1 language<br>**$0.36** | 2 languages<br>**$0.72** | 15 languages<br>**$5.40** | 27 languages<br>**$9.72** |
| *Your audience subtotal* | **$0.76** | **$1.44** | **$12.60** | **$20.92** |
| **TOTAL — every allowance spent** | **$6.77** | **$12.62** | **$75.13** | **$218.79** |
| <sub>as a share of the price</sub> | <sub>—</sub> | <sub>66% of $19</sub> | <sub>76% of $99</sub> | <sub>73% of $299</sub> |
| <sub>left after payment fees</sub> | <sub>-$6.77</sub> | <sub>$5.40</sub> | <sub>$20.00</sub> | <sub>$69.14</sub> |


### The same numbers without the prose

Costs alone, for scanning and for pasting into a spreadsheet.

In [7]:
# The bare grid: one row per metric, one column per tier, cost only.
rows = [[s(f'usage.metric.{m}')] + [usd(cost_of(m, PLANS[t]['caps'].get(m))) for t in TIERS]
        for m in METRICS]
rows.append(['**Total**'] + [f'**{usd(maxed[t])}**' for t in TIERS])
table(['Service'] + [s('plan.tier.' + t) for t in TIERS], rows, 'Cost only')

# Which rows actually matter: on the largest tier, a handful carry nearly all of it.
top = sorted(METRICS, key=lambda m: -cost_of(m, PLANS['max']['caps'].get(m)))
running = 0
for m in top[:5]:
    c = cost_of(m, PLANS['max']['caps'].get(m))
    running += c
    print(f"{s('usage.metric.' + m):<26} {usd(c):>8}  ({c / maxed['max']:>5.1%})  "
          f"running {running / maxed['max']:.0%}")

**Cost only**

| Service | Free | Fresh | Pro | Max |
|---|---|---|---|---|
| Audio recording time | $1.20 | $2.40 | $9.60 | $52.80 |
| AI generation | $1.69 | $2.53 | $21.94 | $37.12 |
| Narration | $0.96 | $1.92 | $12.80 | $22.40 |
| Premium narration | $0.60 | $1.20 | $3.00 | $13.20 |
| Speaker identification | $1.20 | $2.40 | $9.60 | $52.80 |
| Translation | $0.18 | $0.36 | $2.00 | $8.00 |
| AI images | $0.17 | $0.34 | $3.36 | $10.08 |
| Image searches | $0.00 | $0.00 | $0.00 | $0.00 |
| Stored audio | $0.01 | $0.02 | $0.16 | $1.13 |
| Imports | <$0.01 | $0.01 | $0.04 | $0.20 |
| Exports | <$0.01 | <$0.01 | $0.04 | $0.14 |
| Narration for viewers | $0.40 | $0.72 | $7.20 | $11.20 |
| Translations for viewers | $0.36 | $0.72 | $5.40 | $9.72 |
| **Total** | **$6.77** | **$12.62** | **$75.13** | **$218.79** |


Audio recording time         $52.80  (24.1%)  running 24%
Speaker identification       $52.80  (24.1%)  running 48%
AI generation                $37.12  (17.0%)  running 65%
Narration                    $22.40  (10.2%)  running 75%
Premium narration            $13.20  ( 6.0%)  running 82%


## 6. Where these caps come from

`sttMinutes` — the *Audio recording time* row — is the only cap set by hand.
Eight of the others are derived from it: N minutes of recording is N minutes of
lecture, which fixes the slide count, which fixes the tokens to generate and
refine those slides, the characters to narrate them, and the megabytes to keep
their audio.

The arithmetic lives in
[`../scripts/pricing/derive-caps.mjs`](../scripts/pricing/derive-caps.mjs) and
is **invoked** here rather than reimplemented — a second copy in Python is
exactly the kind of thing that silently stops agreeing with what ships.

```bash
npm run caps:derive               # report what would change
npm run caps:derive -- --write    # apply to config/plans.json
npm run caps:check                # exit 1 if a cap has drifted from the model
```

In [8]:
import subprocess

def derive(mode='floor', stt=None):
    """Runs the cap deriver and returns {tier: {metric: cap}}.

    One implementation, called rather than copied: whatever ships is what is
    charted here. `stt` previews a different recording allowance — {'pro': 1950}
    — without touching config/plans.json.
    """
    cmd = ['node', os.path.join(ROOT, 'scripts', 'pricing', 'derive-caps.mjs'),
           '--json', f'--mode={mode}']
    for tier, minutes in (stt or {}).items():
        cmd += ['--stt', f'{tier}={minutes:.0f}']
    out = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    if out.returncode:
        raise SystemExit(f'derive-caps.mjs failed: {out.stderr.strip()}')
    return json.loads(out.stdout)


fit = derive('fit')       # what the recording allowance alone implies
DERIVED = list(fit['free'])

# Shipped vs implied. A tier ships the LARGER of the two ('floor' mode), so a
# gap means the cap was sized for a lecture volume bigger than the recording
# allowance rations — which is deliberate on the tiers where recording is
# rationed hardest.
rows = []
for metric in DERIVED:
    cells = []
    for tier in TIERS:
        shipped, implied = PLANS[tier]['caps'][metric], fit[tier][metric]
        cells.append(f'{shipped:,}' if shipped == implied
                     else f'{shipped:,} <sub>(recording implies {implied:,})</sub>')
    rows.append([s(f'usage.metric.{metric}')] + cells)
table(['Derived cap'] + [s('plan.tier.' + t) for t in TIERS], rows,
      'Shipped, and what the recording allowance on its own would justify')

print('Recording allowance: ' +
      ' | '.join(f"{t} {PLANS[t]['caps']['sttMinutes']:,} min "
                 f"(~{PLANS[t]['caps']['sttMinutes'] / A['lecture']['durationMinutes']:.0f} lectures)"
                 for t in TIERS))

**Shipped, and what the recording allowance on its own would justify**

| Derived cap | Free | Fresh | Pro | Max |
|---|---|---|---|---|
| AI generation | 5,000,000 <sub>(recording implies 2,500,000)</sub> | 7,500,000 <sub>(recording implies 4,900,000)</sub> | 65,000,000 <sub>(recording implies 20,000,000)</sub> | 110,000,000 |
| Narration | 60,000 <sub>(recording implies 30,000)</sub> | 120,000 <sub>(recording implies 60,000)</sub> | 800,000 <sub>(recording implies 240,000)</sub> | 1,400,000 |
| Premium narration | 20,000 <sub>(recording implies 9,900)</sub> | 40,000 <sub>(recording implies 20,000)</sub> | 100,000 <sub>(recording implies 80,000)</sub> | 440,000 |
| Speaker identification | 75 | 150 | 600 | 3,300 |
| Translation | 9,000 | 18,000 | 100,000 <sub>(recording implies 72,000)</sub> | 400,000 |
| Stored audio | 500 <sub>(recording implies 220)</sub> | 1,000 <sub>(recording implies 440)</sub> | 8,000 <sub>(recording implies 1,800)</sub> | 58,000 |
| Narration for viewers | 25,000 <sub>(recording implies 14,000)</sub> | 45,000 <sub>(recording implies 27,000)</sub> | 450,000 <sub>(recording implies 110,000)</sub> | 700,000 <sub>(recording implies 600,000)</sub> |
| Translations for viewers | 1 | 2 <sub>(recording implies 1)</sub> | 15 <sub>(recording implies 4)</sub> | 27 <sub>(recording implies 22)</sub> |


Recording allowance: free 75 min (~1 lectures) | fresh 150 min (~2 lectures) | pro 600 min (~8 lectures) | max 3,300 min (~44 lectures)


### Change the recording allowance and everything follows

The point of deriving rather than hand-setting: moving one number moves the
eight that depend on it, and the cost moves with them. Below, each tier's
recording allowance is re-run at half, today's, and double — every other cap
re-derived from scratch each time.

The `fit` sizing is used here, since the question is what a given allowance
implies rather than what a tier happens to ship on top of it.

In [9]:
# Re-derive every tier at several recording allowances and price the result, so
# the cost consequence of moving the one hand-set number is visible.
MULTIPLES = [0.5, 1.0, 2.0]

rows = []
for tier in TIERS:
    base = PLANS[tier]['caps']['sttMinutes']
    cells = []
    for factor in MULTIPLES:
        minutes = base * factor
        caps = dict(PLANS[tier]['caps'])
        caps.update(derive('fit', {tier: minutes})[tier])
        caps['sttMinutes'] = minutes
        total = sum(cost_of(m, caps.get(m)) for m in caps)
        price = price_of(tier)
        cells.append(f'{minutes:,.0f} min → {usd(total)}'
                     + (f' <sub>({total / price:.0%} of price)</sub>' if price else ''))
    rows.append([f"**{s('plan.tier.' + tier)}**"] + cells)

table(['Tier'] + [f'{f:g}× recording' for f in MULTIPLES], rows,
      'Worst-case cost as the recording allowance moves, everything else re-derived')

# The shipped Pro caps were originally sized for 26 lectures a month. Deriving
# from that many minutes should reproduce them — a check that the formula
# matches the reasoning the caps were actually built on.
lectures = A['tiers']['pro']['lecturesPerMonth']
implied = derive('fit', {'pro': lectures * A['lecture']['durationMinutes']})['pro']
print(f"Pro re-derived at {lectures} lectures "
      f"({lectures * A['lecture']['durationMinutes']:,} min) vs shipped:")
for metric in ('aiTokens', 'ttsCharacters', 'audienceLocales'):
    print(f"  {metric:<22} derived {implied[metric]:>12,}   shipped {PLANS['pro']['caps'][metric]:>12,}")

**Worst-case cost as the recording allowance moves, everything else re-derived**

| Tier | 0.5× recording | 1× recording | 2× recording |
|---|---|---|---|
| **Free** | 38 min → $2.79 | 75 min → $4.96 | 150 min → $9.35 |
| **Fresh** | 75 min → $5.14 <sub>(27% of price)</sub> | 150 min → $9.52 <sub>(50% of price)</sub> | 300 min → $18.69 <sub>(98% of price)</sub> |
| **Pro** | 300 min → $21.79 <sub>(22% of price)</sub> | 600 min → $40.31 <sub>(41% of price)</sub> | 1,200 min → $76.95 <sub>(78% of price)</sub> |
| **Max** | 1,650 min → $112.73 <sub>(38% of price)</sub> | 3,300 min → $215.39 <sub>(72% of price)</sub> | 6,600 min → $418.85 <sub>(140% of price)</sub> |


Pro re-derived at 26 lectures (1,950 min) vs shipped:
  aiTokens               derived   64,000,000   shipped   65,000,000
  ttsCharacters          derived      780,000   shipped      800,000
  audienceLocales        derived           13   shipped           15


## 7. What this table does not include

It prices **allowances**, so it is bounded by exactly what the caps bound —
which is not everything we pay for.

- **Playback egress.** Viewers streaming a deck's images and audio is not
  metered against any cap, so it appears in no cell. It is small per playback
  and real in aggregate; `billing-cost-model.ipynb` §3 models it per lecture.
- **Fixed monthly costs** — app platform, database, backups, domain, tooling.
  Per-subscriber costs cannot cover these on their own; break-even is
  `billing-cost-model.ipynb` §6.
- **Payment fees** are shown only in the summary row, since they are charged per
  subscription rather than per allowance.
- **Vendor free tiers** are ignored on purpose. Google gives a monthly free
  allowance per *account* — 1M TTS characters, 500k translated characters — not
  per user of ours. Spread across subscribers it rounds to nothing, and counting
  on it would understate the marginal cost of the next subscriber, which is the
  number a price has to survive.

And two things the caps deliberately do not stop:

- **Storage accrues.** `Imports` and `Stored audio` are priced here for one
  month of holding. Imported seed documents are kept for reprocessing with no
  retention sweep, so that cost repeats every month after the period that
  incurred it. Recorded audio does not: the retention row above deletes it.
- **The worst case is not the expected case.** Nobody spends every allowance to
  zero. Utilisation levels are modelled in `billing-cost-model.ipynb` §5; the
  planning case is roughly half of what is shown here.